In [ ]:
from pyspark.sql.functions import col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    parse_timestamp,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_order_reviews_table_name = dbutils.widgets.get("raw_olist_order_reviews_table")

silver_schema = dbutils.widgets.get("silver_schema")
order_reviews_table_name = dbutils.widgets.get("order_reviews_table")

In [ ]:
raw_olist_order_reviews_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_order_reviews_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{order_reviews_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{order_reviews_table_name} (
            orderId STRING,
            reviewId STRING,
            reviewScore INT,
            reviewCommentTitle STRING,
            reviewCommentMessage STRING,
            reviewCreationTimestamp TIMESTAMP,
            reviewAnswerTimestamp TIMESTAMP,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
order_reviews_silver_df = with_processed_timestamp(
    raw_olist_order_reviews_df.where(is_valid_uuid("order_id") & is_valid_uuid("review_id"))
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("review_id").cast("string").alias("reviewId"),
        col("review_score").cast("int").alias("reviewScore"),
        col("review_comment_title").cast("string").alias("reviewCommentTitle"),
        col("review_comment_message").cast("string").alias("reviewCommentMessage"),
        parse_timestamp("review_creation_date").alias("reviewCreationTimestamp"),
        parse_timestamp("review_answer_timestamp").alias("reviewAnswerTimestamp"),
    )
    .dropDuplicates(["orderId", "reviewId"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{order_reviews_table_name}",
    source_view="order_reviews_silver_view",
    keys=["orderId", "reviewId"],
    source_df=order_reviews_silver_df,
)